In [1]:
import torch
import torch.nn as nn
import math
from collections import Counter

text = """
機器學習是人工智慧的重要領域。
深度學習可以從大量資料中學習規律。
Transformer 是一種處理序列資料的模型。
它使用注意力機制來理解文字之間的關係。
語言模型可以根據前面的文字預測下一個字。
學生可以透過簡單範例理解 Transformer 的基本架構。
"""

In [2]:
counter = Counter(text)
chars = [ch for ch, count in counter.most_common()] # 依照出現次數排序
stoi = {ch: i for i, ch in enumerate(chars)} # 建立文字到數字的對應
itos = {i: ch for ch, i in stoi.items()} # 建立數字到文字的對應

vocab_size = len(chars)

data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long) # 將文字轉成數字編碼

print("vocab size:", vocab_size)
print("data shape:", data.shape)
print(chars)
print(stoi['機'])
print(itos[stoi['器']])
print(data)

vocab size: 82
data shape: torch.Size([134])
['\n', '。', 'r', '的', '學', '習', '可', '以', ' ', '理', '字', '機', '是', '資', '料', 'T', 'a', 'n', 's', 'f', 'o', 'm', 'e', '一', '模', '型', '解', '文', '器', '人', '工', '智', '慧', '重', '要', '領', '域', '深', '度', '從', '大', '量', '中', '規', '律', '種', '處', '序', '列', '它', '使', '用', '注', '意', '力', '制', '來', '之', '間', '關', '係', '語', '言', '根', '據', '前', '面', '預', '測', '下', '個', '生', '透', '過', '簡', '單', '範', '例', '基', '本', '架', '構']
11
器
tensor([ 0, 11, 28,  4,  5, 12, 29, 30, 31, 32,  3, 33, 34, 35, 36,  1,  0, 37,
        38,  4,  5,  6,  7, 39, 40, 41, 13, 14, 42,  4,  5, 43, 44,  1,  0, 15,
         2, 16, 17, 18, 19, 20,  2, 21, 22,  2,  8, 12, 23, 45, 46,  9, 47, 48,
        13, 14,  3, 24, 25,  1,  0, 49, 50, 51, 52, 53, 54, 11, 55, 56,  9, 26,
        27, 10, 57, 58,  3, 59, 60,  1,  0, 61, 62, 24, 25,  6,  7, 63, 64, 65,
        66,  3, 27, 10, 67, 68, 69, 23, 70, 10,  1,  0,  4, 71,  6,  7, 72, 73,
        74, 75, 76, 77,  9, 26,  8, 15,  2, 16, 17, 18, 19

In [3]:
block_size = 16
batch_size = 8

def get_batch():
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

In [4]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [5]:
class TinyTransformerLM(nn.Module):
    def __init__(self, vocab_size, d_model=64, nhead=4, num_layers=2):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=128,
            dropout=0.1,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoder(x)
        x = self.transformer(x)
        logits = self.fc(x)
        return logits

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = TinyTransformerLM(vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for step in range(1000):
    x, y = get_batch()
    x = x.to(device)
    y = y.to(device)

    logits = model(x)

    loss = loss_fn(
        logits.reshape(-1, vocab_size),
        y.reshape(-1)
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(step, loss.item())

0 4.640064716339111
100 0.5771522521972656
200 0.12532192468643188
300 0.06470149010419846
400 0.03243635594844818
500 0.02856016345322132
600 0.01500575803220272
700 0.011119838804006577
800 0.005924365017563105
900 0.0059206788428127766


In [10]:
def generate(model, start_text, max_new_tokens=80):
    model.eval()

    ids = [stoi[ch] for ch in start_text]
    x = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(device)

    for _ in range(max_new_tokens):
        x_cond = x[:, -block_size:]

        logits = model(x_cond)
        next_logits = logits[:, -1, :]
        probs = torch.softmax(next_logits, dim=-1)

        next_id = torch.multinomial(probs, num_samples=1) #依照機率分佈抽樣
        x = torch.cat([x, next_id], dim=1)

    result = "".join(itos[i] for i in x[0].tolist())
    return result

print(generate(model, "機器學習", 80))

機器學習規律。
深度學習規律。
Transfosformer 的基本架構。
。
。
or 是一種處理序列資料的模型。
它使用注意力機制來理解文字之間的關係。
語言模型可
